# firelab — User Guide

How to run firelab, use the control panel, drive a run from code, and fix common
problems.

| Notebook | Answers |
|---|---|
| `ARCHITECTURE.ipynb` | **How is it built?** Components, data flow, every rule and constant, the API, the tests. |
| `GUIDE.ipynb` *(this one)* | **How do I use it?** Starting the services, every panel, a first run, scripted sessions, troubleshooting. |

| § | Section |
|---|---|
| 1 | Starting the services |
| 2 | The control panel, panel by panel |
| 3 | A guided first run |
| 4 | Working with sensors and faults |
| 5 | Driving a session from this notebook |
| 6 | Replacing the rule detector with a trained model |
| 7 | What the simulation does not do |
| 8 | Troubleshooting |

## 1. Starting the services

Start two services, in this order.

**BuildSim** holds the building (floor plans, walkable graph, 3D viewer). Run it
from the `buildingsim/` folder next to `firelab/`:

```bash
cd buildingsim
make run                                     # builds if needed, then starts on 127.0.0.1:9090
```

Its request log is very chatty. To keep it out of your terminal, run it in the
background with the output sent to a file:

```bash
make run > /tmp/buildsim.log 2>&1 &
```

**firelab** runs the simulation and serves the control panel:

```bash
cd firelab
make build      # .venv + Python deps, then npm install + vite build into ui/dist
make run        # uvicorn on 127.0.0.1:8090
```

Open <http://127.0.0.1:8090/> for the control panel and <http://127.0.0.1:9090/>
for the 3D viewer, side by side. firelab reads the building from BuildSim when it
starts, so **start BuildSim first**. If you start firelab first, press
**Reload floors** once BuildSim is up.

### Makefile targets

| Target | What it does |
|---|---|
| `make deps` | create `.venv` (with `uv` if installed, else `python3 -m venv`) and install `requirements.txt` |
| `make ui` | `npm install` and `npm run build` in `ui/` |
| `make build` | `deps` + `ui` |
| `make run` | serve the API and the built UI on port 8090 |
| `make dev` | same, with `--reload` for Python edits; run `cd ui && npm run dev` beside it for hot-reloading UI work on port 5173 |
| `make stop` | stop a detached firelab server |
| `make test` | run the test suite (needs no services) |
| `make clean` | remove `.venv`, `ui/dist`, `ui/node_modules`, `__pycache__` |

### Restarting cleanly

The live event stream never closes on its own, so `make run` passes
`--timeout-graceful-shutdown 3` to let the server exit. To restart a detached
server:

```bash
make stop && make run
```

**Do not run `occupancysim` at the same time.** Both it and firelab replace the
whole `PUT /api/entities` and `PUT /api/occupancy` collections, so they erase each
other's people.

### Checking both services from this notebook

The cell below defines the `api()` helper that section 5 uses, and checks that both
services answer.

In [ ]:
import json
import sys
import urllib.request
from pathlib import Path

FIRELAB = 'http://127.0.0.1:8090'    # the simulation + control panel
BUILDSIM = 'http://127.0.0.1:9090'   # the building and 3D viewer

# The notebook lives in firelab/, so the package root is one level up.
ROOT = Path.cwd() if Path.cwd().name == 'firelab' else Path.cwd() / 'firelab'
sys.path.insert(0, str(ROOT.parent))  # makes `import firelab...` work (section 6)


def api(path, method='GET', body=None, base=FIRELAB):
    '''Send one HTTP request and return the decoded JSON reply.'''
    data = json.dumps(body).encode() if body is not None else None
    request = urllib.request.Request(
        base + path, data=data, method=method,
        headers={'content-type': 'application/json'},
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.loads(response.read() or 'null')


try:
    print('firelab  ', api('/healthz'))
    building = api('/api/building', base=BUILDSIM)
    print('buildsim ', building['name'], [level['id'] for level in building['levels']])
except Exception as exc:
    print('not reachable:', exc)
    print('start both services as described above, then re-run this cell')

## 2. The control panel, panel by panel

The UI is laid out in three columns. Everything on screen comes from one live
stream (`/api/events`), so all panels always show the same moment. A slider or
number field keeps your value while you edit it and sends it when you release or
apply.

```
Header ─────────────────────────────────────────────────────────────
 Scenario    │  Rooms                              │  Response
 Scorecard   │  (truth · reading · belief)         │  Life safety
 Evacuation  │                                     │  Simulation settings
 Population  │  Room detail (charts + Why NN%)     │  Journal
 Sensors     │                                     │
```

### Header

| Control | What it does |
|---|---|
| **BuildSim online/offline** | whether firelab can currently talk to BuildSim |
| **N spaces** | rooms loaded from the floor plans (956 in the provided building) |
| **live / reconnecting** | whether the browser's event stream is connected; if it says *reconnecting*, the numbers on screen are frozen |
| clock | simulated time of day, starting at 08:00:00 |
| **Run / Pause** | starts and stops simulated time |
| speed slider | 1×–60× simulated seconds per real second (sent when you release it) |
| **Reload floors** | re-read the floor plans from BuildSim and reconnect; use it after restarting BuildSim |
| **Export CSV** | download every recorded sample, labelled with the source kind |
| **Reset** | clear the run: fires, physics, alarms, doors, sprinklers, faults, scorecard and history; keeps floor plans, deployed sensors, settings and the clock |
| **3D viewer ↗** | open BuildSim in a new tab |

### Scenario

1. Pick a **Room** with the search box. Every other control here needs a room.
2. Either press a **preset**, or build a source by hand and press **Ignite**.

A preset deploys smoke, CO and temperature sensors in the room **and its
neighbours**, optionally breaks that room's smoke detector, then ignites:

| Preset | Source | Expected result |
|---|---|---|
| Office fire | fast flaming fire | confirmed quickly |
| Overnight smoulder | smouldering | confirmed on CO; the sprinkler is refused because the room stays below 35 °C |
| Kitchen cooking | cooking | no alarm (any alarm is a false alarm) |
| Dust storm | dust | no alarm |
| Fire with a dead detector | fast flaming fire, smoke detector `dead` | still confirmed on CO and heat, but later |
| Drifting sensor, no fire | nothing, smoke detector `drift` | no alarm |

**Manual ignition:** choose the **Source** (flaming, smouldering, cooking, dust,
steam). For flaming fires you also set **Growth** (slow → ultrafast) and **Peak
(kW)**. **Start delay** schedules the source to light later, for example a
nuisance that arrives mid-fire. Manual ignition does **not** deploy sensors.

Active sources are listed at the bottom. **remove** puts one out; the heat and
smoke it already made still have to clear.

### Scorecard

The run grades itself, because every source knows whether it is a real fire:

- **caught / missed / false alarms**: a fire counts as missed if it is not confirmed within 5 minutes of ignition. An alarm caused by smoke drifting in from a real fire next door is not counted as false.
- **Mean time to confirm** and **Slowest**: ignition → CONFIRMED.
- **Ignition to alarm**, **Alarm to building clear**, **RSET, total**: the evacuation timeline.
- Recent false alarms are listed with their cause (e.g. `cooking`, or `none` for a faulty sensor).

### Evacuation

One row per floor: **Inside**, **Moving**, **Stranded**, **Out**, plus a total.
**Stranded** people have status *no route*: BuildSim returned no path from their
room to any exit. Their rooms are named underneath.

### Population

How many of each role are in the building: students, lecturers, staff, security,
visitors. Each role walks at its own **Speed** factor, and **Out** shows
safe/total. Edit the counts and press **Apply & scatter**. This re-places
**everyone**, including people who already got out, so do it before igniting.

### Sensor deployment

Filter by name and floor, tick rooms (or **Select all (N)** to take every match),
choose modalities, then press **Deploy**. **Remove** takes the sensors out of the
ticked rooms again. The hint line shows the devices deployed and how many are
faulty. Deployed sensors also appear in BuildSim's equipment tree.

A room without sensors has no readings and no agent, so **it can never raise an
alarm**, however hard it burns.

### Rooms

The main table. It lists rooms that have sensors, plus any room with a source or
a P(fire) above 5 %. A room with a source but no sensors shows state `UNMONITORED`.

| Columns | Meaning |
|---|---|
| **Truth** °C / smoke / ppm | what the simulation says is really in the room |
| **Reading** °C / smoke / ppm | what the sensors report (`—` = no reading yet) |
| **P(fire)** | the detector's probability |
| **State** | `NORMAL`, `INVESTIGATING`, `PRE_ALARM`, `CONFIRMED`, `SUPPRESSED`, `CLEARING` |
| **Response** | 💧 sprinkler on · 🚪 door closed · N👤 people in the room |

A gap between truth and reading is sensor error. A gap between reading and state
is a detector decision.

- Rows are sorted by P(fire), highest first.
- The floor dropdown shows how many listed rooms each floor has.
- **only active** hides rooms that are `NORMAL` with P(fire) ≤ 10 %.
- **Clicking a row** selects it (for Room detail and Response) and expands its
  device list. **Clicking it again** collapses the list. Each device shows its
  latest reading and a **fault** dropdown (section 4).

### Room detail

For the selected room:

- four small charts: **smoke**, **temperature** and **CO** (grey = truth,
  coloured = reading) and **P(fire)**. Gaps in a line are missing samples.
  History covers the last 240 samples, one every 5 simulated seconds (about 20 minutes).
- **Why NN%**: the detector's weighted terms, largest first. Red bars push
  towards fire, blue bars push away (the `bias` term is always negative). Hover a
  number to see the raw feature value.

The panel refreshes every 2 s while running, and every 6 s while paused.

### Response

- **Autonomous response**: when unticked, the agent still changes state and raises
  alarms, but its commands are not carried out. Use it to show what the system
  *would* do.
- Counters: alarms, evacuating, safe, occupants.
- **Manual override** for the selected room: **Sprinkler on/off**, **Door
  close/open**, **Evacuate**. These pass through the same safety interlocks as the
  agent's commands, and the verdict is printed (`allowed — …` or `blocked — reason`).
- **Thresholds**: sliders for **Investigate**, **Pre-alarm** and **Confirm**
  (0.05–0.95), applied live. Lower Confirm and false alarms start appearing in the
  Scorecard.

The interlocks are:
- a sprinkler needs the room at **≥ 35 °C**;
- fire doors never lock;
- a door is not closed while people are in the room or the room is on someone's escape route.

### Life safety

- **people exposed**: people standing in rooms that are no longer escapable (≥ 60 °C, or smoke that cuts visibility to about 10 m or less).
- **over CO dose**: people whose accumulated CO dose passed the incapacitation limit (FED 0.3). This counts people who already got out.
- **rooms lost**: the number of untenable rooms. The worst are listed with the reason and headcount.
- **Worst dose taken**: the highest dose so far, as a percentage of an incapacitating one.

### Simulation settings

| Setting | Range | Takes effect |
|---|---|---|
| **Sensor noise** | 0× (perfect) – 3× (poor) | **for sensors deployed after the change**: remove and redeploy to apply it to existing ones |
| **Sampling interval** | 1–60 s | same as noise: new deployments only |
| **Walking speed** | 0.5–2 m/s | immediately; each role's factor multiplies it |
| **Random seed** | integer | immediately restarts the random number stream |

For two runs to be comparable, restart the random stream before each one: send the
seed, press **Reset**, then set up the same scenario. The field only sends when its
value changes, so to reuse the same seed, type another number and then the original
one again (or call `PUT /api/config` with `{"seed": N}`). Note: the panel's hint says
the seed matters "from the next reset", but the server actually restarts the random
stream as soon as the seed is sent.

### Journal

Newest first: state changes (`agent`), actuator commands (`actuator`), refused
commands (`interlock`), injected faults (`fault`), scenario changes, and errors.
It keeps the last 40 entries on screen.

## 3. A guided first run

Both services must be running (section 1). Open <http://127.0.0.1:8090>.

**1 — Check the header.** It should show **BuildSim online**, **956 spaces** and
**live**. If BuildSim is offline, start it and press **Reload floors**.

**2 — Deploy sensors.** In **Sensor deployment**, set the floor to `level0`, press
**Select all (318)**, leave all three modalities ticked, and press **Deploy**. The
Rooms table fills with 318 rows, and the detectors appear in BuildSim's equipment
tree.

**3 — Start the clock.** Press **Run** and drag the speed to about **20×**. Nothing is
burning yet, so every P(fire) should stay near 0 %. This is your no-false-alarm
baseline.

**4 — Light a fire.** In **Scenario**, search for a `level0` room and select it,
then press the **Office fire** preset. The source appears in the list at the
bottom of the panel.

**5 — Watch the room turn over.** Tick **only active** in the Rooms table. The
burning room rises to the top, because rows are sorted by P(fire). Things happen
in this order:

> truth rises → reading follows, a little late and a little wrong → P(fire) climbs →
> state `INVESTIGATING` → `PRE_ALARM` → `CONFIRMED`

On `CONFIRMED` the agent asks for the sprinkler, a closed fire door and an
evacuation. In the 3D viewer, the room is outlined in red, people start walking to
the exits, and one escape route is drawn.

**6 — Open the room.** Click its row. **Room detail** charts truth against reading,
and **Why NN%** shows which terms produced the probability.

**7 — Look at the response.** The Response column and the Journal show what was
done, and which commands the interlocks refused. For example, the door is refused
while people are still inside. Untick **Autonomous response** and try a manual
command to see the interlock verdict. The **Scorecard** records the detection
and its time to confirm.

**8 — Break something on purpose.** See section 4.

**9 — Save the evidence.** **Export CSV** downloads one row per sample per monitored
room, with the true source kind as the label.

> **Reset** clears the fires, alarms, doors, faults, scorecard and history, but keeps
> your sensors, your settings and the clock. **Reload floors** re-reads the building
> from BuildSim; you only need it after restarting BuildSim.

## 4. Working with sensors and faults

A sensor is the only link between the true state of a room and what the system
believes about it.

### Deploying

A **device** is one modality in one room. Deploying all three modalities to every
room gives 956 × 3 = **2868 devices**.

| Modality | Measures | Response time | Character |
|---|---|---|---|
| `smoke` | obscuration (1/m) | fastest (τ 12 s) | fooled by dust, steam and cooking |
| `co` | carbon monoxide (ppm) | τ 25 s | the best sign of real combustion; catches smouldering fires |
| `temperature` | air temperature (°C) | slowest (τ 30 s) | hardest to fake; needed for the sprinkler |

The detector trusts a room less when fewer modalities report. With **only one**
modality, P(fire) can reach at most 70 %, below the 75 % confirm threshold, so a
room with a single sensor type **can never confirm a fire**.

### What a reading goes through

Every device turns the true value into a reading in this order:

> lag → fault → calibration bias → noise → rounding → sensor range

It only reports once every sampling interval (5 s by default). So a reading is
late, noisy, rounded and possibly broken, which is why truth and reading are
charted side by side.

### Injecting faults

In the **Rooms** table, click a room to expand its devices. Each device has a
fault dropdown, which you can change mid-run without pausing:

| Fault | Behaviour | What to watch for |
|---|---|---|
| `none` | healthy | the baseline |
| `stuck` | repeats its last value forever | a flat reading while truth changes |
| `dropout` | randomly skips 40 % of its samples | a slower, gappier trace |
| `drift` | error grows a little with every sample | slow divergence from truth |
| `dead` | never reports again | once its old readings age out of the 120 s window, that modality stops counting |

Faults are cleared by **Reset**. Two presets show fault handling:

- **Fire with a dead detector**: a real fire with the smoke detector dead. CO and
  temperature still confirm it, but later than the Office fire.
- **Drifting sensor, no fire**: nothing is burning while the smoke detector drifts
  upwards. The other modalities do not agree, so no alarm is raised.

### Experiments worth running

1. **One lying sensor.** Deploy all three modalities to a room, set its `smoke`
   device to `drift`, light nothing, and let it run. Then set `co` to `drift` too,
   and compare how far P(fire) climbs.
2. **Losing a sense mid-fire.** Ignite a flaming fire. Just as P(fire) starts to
   climb, set one modality to `dead`. Watch whether the other two still reach
   CONFIRMED, and use **Why NN%** to see which term lost its contribution.
3. **Noisier hardware.** Set **Sensor noise** to 3×, remove and redeploy the
   sensors, and run the Kitchen cooking and Office fire presets again. Compare
   the Scorecard.

## 5. Driving a session from this notebook

Everything the control panel does is an HTTP call, so a run can be scripted and
repeated for a report. Run the helper cell in section 1 first.

> **These cells change the live simulation:** they reset it and deploy sensors to
> every room. Watch the control panel and 3D viewer while they run. The plotting
> cell needs matplotlib in the notebook's kernel (`pip install matplotlib`; it is
> not in firelab's `.venv`).

The first cell:

1. resets the run and fixes the seed;
2. deploys all three modalities to every room;
3. lights a smouldering fire (the hard case) in a room on `level1`;
4. runs at 40× until the fire is detected, or about 75 real seconds pass;
5. pauses and prints the scorecard and evacuation state.

In [ ]:
import time

# 1. A repeatable start: fix the seed, then clear the previous run.
api('/api/config', 'PUT', {'seed': 1})
api('/api/reset', 'POST')

# 2. All three sensor types in every room (same as Select all + Deploy).
rooms = api('/api/rooms')
api('/api/sensors/deploy', 'POST', {
    'spaces': [room['key'] for room in rooms],
    'modalities': ['smoke', 'co', 'temperature'],
})
print(len(rooms), 'rooms instrumented')

# 3. A decent-sized room on the middle floor.
target = next(r for r in rooms
              if r['level'] == 'level1' and r['kind'] == 'room' and r['area'] > 25)
print('igniting in', target['key'])
api('/api/scenario/ignite', 'POST', {'space': target['key'], 'kind': 'smouldering'})

# 4. Run at 40x and watch that room every 3 s until it is detected.
api('/api/clock', 'POST', {'factor': 40, 'running': True})
for _ in range(25):
    time.sleep(3)
    state = api('/api/state')
    room = next(s for s in state['spaces'] if s['key'] == target['key'])
    print(f"{state['clock']['text']}  p={room['p_fire']:.2f}  {room['state']:<14}"
          f"  smoke={room['truth']['smoke']:.2f}  CO={room['truth']['co']:.0f}ppm")
    if state['score']['detections']:
        break

# 5. Pause and report.
api('/api/clock', 'POST', {'running': False})
print()
print('score:', state['score'])
print('evacuation:')
for level in state['evacuation']['levels']:
    print('  ', level)
print('stranded:', state['evacuation']['stranded'])

The recorded history of that room: the same data the Room detail panel charts.

In [ ]:
import urllib.parse

import matplotlib.pyplot as plt

# The '/' in a room key must be percent-encoded.
track = api('/api/history?space=' + urllib.parse.quote(target['key'], safe=''))
columns = track['columns']
print(len(track['points']), 'samples')

if track['points']:
    def column(name):
        index = columns.index(name)
        return [point[index] for point in track['points']]

    t0 = track['points'][0][0]
    minutes = [(t - t0) / 60 for t in column('t')]

    # Truth (grey) against reading (coloured); the gap is sensor error.
    figure, axes = plt.subplots(1, 4, figsize=(16, 3.2))
    for axis, (field, label) in zip(axes, [('smoke', 'smoke (1/m)'),
                                           ('temperature', 'temperature (°C)'),
                                           ('co', 'CO (ppm)')]):
        axis.plot(minutes, column('truth_' + field), color='0.65', label='truth')
        axis.plot(minutes, column('read_' + field), label='reading')
        axis.set_title(label)
        axis.set_xlabel('minutes')
        axis.legend(fontsize=8)

    axes[3].plot(minutes, column('p_fire'), color='crimson')
    axes[3].axhline(0.75, ls='--', lw=0.8, color='0.5')
    axes[3].set_title('P(fire), confirm threshold dashed')
    axes[3].set_xlabel('minutes')
    figure.tight_layout()
    plt.show()

Why the detector reached its current probability: the terms behind the **Why NN%** bars.

In [ ]:
# `value` is the raw feature; `weighted` is its push towards (+) or away from (-) fire.
for term in track['why']:
    bar = '#' * int(abs(term['weighted']) * 8)
    print(f"{term['name']:<22} raw={term['value']:>10.3f}   {term['weighted']:+6.2f}  {bar}")

What **Export CSV** downloads: one row per sample per monitored room, with a
`label` column holding the true source kind for that room (`none` if nothing was lit there).

In [ ]:
with urllib.request.urlopen(FIRELAB + '/api/export.csv', timeout=120) as response:
    csv_text = response.read().decode()

rows = csv_text.splitlines()
print(rows[0])           # header: space, level, room, label, t, truth_*, read_*, p_fire
for row in rows[1:4]:
    print(row)
print('...')
print(len(rows) - 1, 'rows')

## 6. Replacing the rule detector with a trained model

The installed detector is `FusionRule`, a hand-weighted rule in
`domain/detector.py`. It is the baseline a trained model has to beat. Everything
downstream (agent, interlocks, Why bars, scorecard) uses the `Detector`
protocol, so swapping the detector touches one line.

**Step 1 — collect labelled data.** Run a variety of presets and manual scenarios,
with and without faults, and press **Export CSV** after each (history holds about
20 minutes per room, so export before it rolls over). Each row carries the true
source kind as its label. `flaming` and `smouldering` are fires, everything else is not.

Note: the CSV holds readings and P(fire), not the full feature vector. To train on
the detector's exact inputs, rebuild `Features` from the readings, or collect
`firelab.domain.features.extract(...)` results in a script.

**Step 2 — implement the protocol.** It needs a `name`, `probability(features)`
returning 0..1, and `explain(features)` returning one `Contribution` per term:

```python
from firelab.domain.detector import Contribution

NAMES = ['smoke', 'co', 'temperature_rise', 'smoke_rate', 'co_smoke_ratio', 'neighbour_agreement']


class TreeDetector:
    name = 'gbdt'

    def __init__(self, model):
        self._model = model

    def probability(self, features):
        if features.coverage == 0:
            return 0.0                      # no working sensor: no opinion
        return float(self._model.predict_proba([features.vector()])[0][1])

    def explain(self, features):
        # e.g. SHAP values, one per feature, in the same order as features.vector()
        weights = shap_values(self._model, features.vector())
        return [Contribution(n, v, w) for n, v, w in zip(NAMES, features.vector(), weights)]
```

**Step 3 — install it.** In `app/engine/state.py`, change

```python
self.detector = FusionRule()  # swap this line to try a trained model
```

to construct your detector.

Two rules to keep:

- **Do not put the model class in `domain/`.** `tests/test_layering.py` fails if
  any `domain/` module imports outside the standard library (e.g. scikit-learn).
  Put it in `app/`, and add its packages to `requirements.txt`.
- **Keep `explain` honest.** If its terms do not account for the probability, the
  Why bars stop meaning anything.

Compare the two detectors with the same seed and the same scenarios, reading the
Scorecard: detections, misses, false alarms and time to confirm.

## 7. What the simulation does not do

These limits matter when you interpret a run.

### Routing acts on belief, not on truth

A room is only treated as dangerous if its agent is in `PRE_ALARM`, `CONFIRMED` or
`SUPPRESSED`, and only rooms with sensors have agents. **An unmonitored room is
never avoided**, so people can be routed straight through a burning corridor that
has no sensors. The Life safety panel still shows that room as untenable, because
it judges from truth.

Presets only instrument the fire room and its neighbours. If you want evacuation
to avoid smoke-filled corridors, deploy sensors along the escape routes, or to the
whole floor.

### Corridors are not blocked in BuildSim's router

Danger rooms are sent to BuildSim as `blocked`, and BuildSim then refuses to route
anyone *into* them. That only works for rooms with an entry node. Corridors have
none (41 of the 305 named spaces on `level0`), so BuildSim can still return a path
*through* an alarming corridor. firelab asks for a route to every exit and prefers
the one that crosses the fewest danger rooms. A smoky route is still chosen over
no route at all.

### Evacuation is building-wide

One `CONFIRMED` room evacuates **everyone** still inside, on every floor. There is
no phased or zoned evacuation. People in a room in `PRE_ALARM` start moving
earlier.

### Only one route is drawn

The viewer shows a single escape route: the one starting nearest the fire. Every
evacuating person still walks their own route; only the drawn line is shared.

### Reset does not restart everything

Reset keeps the **clock** and whether it is **running**, the **journal**, and the
position in the **random number stream**. To repeat a run exactly, send the seed
again before resetting (see *Simulation settings* in section 2), which restarts the
random stream.

### The physics is simple

Each room is one well-mixed zone. There is no smoke layer, no fuel model and no
ventilation, and a fire only spreads its heat, smoke and CO through doorways and
stairwells. A fire never ignites a neighbouring room. Numbers are plausible, not
predictive.

### The detector is a baseline

`FusionRule` is hand-weighted to separate the six presets, not fitted to data.
Section 6 describes replacing it.

## 8. Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Header shows **BuildSim offline** | BuildSim is not running, or was restarted, or a write failed; firelab stops publishing until it reconnects | start BuildSim, then press **Reload floors** |
| **0 spaces** | firelab started before BuildSim | press **Reload floors** |
| Header shows **reconnecting** | the browser lost the event stream (e.g. firelab restarted) | reload the page |
| No people in the 3D viewer | `occupancysim` is also running and overwrites them | stop it; only one service may write entities |
| No highlights or escape route in the 3D viewer | they are drawn into the most recently active viewer tab | open the viewer tab (and keep it open) before the alarm |
| People vanish | they reached an exit | expected; see **Out** in the Evacuation panel |
| Many people **Stranded** | BuildSim returned no route to any exit, which also happens when BuildSim is unreachable | check BuildSim is up, press **Reload floors** |
| The Rooms table is empty | no sensors deployed and no source lit | deploy sensors (section 3, step 2) |
| A room shows `UNMONITORED` | it has a source but no sensors | deploy sensors to it |
| A reading shows `—` | the sensor has not sampled yet, or it is `dead` | wait one sampling interval, or check the fault dropdown |
| P(fire) never reaches Confirm | only one modality reporting caps P(fire) at 70 % | deploy all three modalities |
| Nothing changes | the clock is paused | press **Run** |
| Alarm but no sprinkler | the interlock refuses water below 35 °C | expected for a smouldering fire; the request stays open and is carried out if the room heats up |
| Door does not close on alarm | people are still in the room, or it is on an escape route | expected; the request stays open until the room is clear |
| Changing noise or interval does nothing | these apply to newly deployed sensors | **Remove** and **Deploy** the sensors again |
| Two runs with the same settings differ | the random stream carried on from the previous run | re-send the seed, then **Reset** (section 2, *Simulation settings*) |
| UI is slow with every room monitored | each snapshot covers ~956 rooms | monitor fewer rooms or lower the speed |
| BuildSim floods the terminal | its request log | redirect it to a file (section 1) |
| `make deps` fails | `python3-venv` is missing | install `uv` or `python3-venv` |
| `No module named matplotlib` in section 5 | not a firelab dependency | `pip install matplotlib` in the notebook's kernel |

### Running the tests

```bash
cd firelab
make test
```

This runs 97 tests in about 2 seconds, with no services needed; the evacuation
tests use a fake BuildSim. The suite covers:
- the domain models;
- the layering rules;
- evacuation routing;
- reset;
- one full run per preset, checking the expected results listed in section 2.

`ARCHITECTURE.ipynb` §15 lists every test file.